# Embedding Evaluation — Example Notebook

End-to-end walkthrough using `example_input.json`.  
Run all cells top-to-bottom; all figures are interactive (Plotly).

The dataset contains 23 crop image embeddings (dim=16) across two classes — `corn` and `soybean` — captured on multiple camera platforms.

In [13]:
import json
import sys
from pathlib import Path

# Ensure the package is importable when running without `pip install -e .`
sys.path.insert(0, str(Path("..").resolve()))

from pai.ag_emb.services.evaluate import run_evaluation
from pai.ag_emb.services.reporting import (
    plot_cosine_similarity,
    plot_knn_confusion,
    plot_lle,
    plot_tsne,
    print_result,
)

## Load Data

In [14]:
# example_input.json lives alongside this notebook in examples/
payload_path = Path("example_input.json")
if not payload_path.exists():
    raise FileNotFoundError(
        "example_input.json not found — start Jupyter from the examples/ directory"
    )

with open(payload_path) as f:
    payload = json.load(f)

embeddings: dict[str, list[float]] = payload["embeddings"]
dim = len(next(iter(embeddings.values())))
print(f"Loaded {len(embeddings)} embeddings  (dim={dim})")

Loaded 23 embeddings  (dim=16)


## Run Evaluation

In [15]:
result = run_evaluation(
    image_embeddings=embeddings,
    k_values=[5, 10],
    dataset_root=None,
    sample_pairs=None,
)

print_result(result)

n_items      : 23
embedding_dim: 16
classes      : ['corn', 'soybean']
k_values     : [5, 10]

── global_metrics ──────────────────────────────────────────────────────
  pairwise cosine    : mean=0.5840  std=0.4318  (p05=0.0455  p50=0.9508  p95=0.9846)
  centroid cosine    : mean=0.7760  std=0.2024  norm=0.7760
  intra/inter gap    : 0.8668  (intra=0.9677  inter=0.1009)

  KNN purity@5      : mean=1.0000  std=0.0000
  KNN purity@10     : mean=0.8783  std=0.1841
  nDCG@5           : mean=1.0000  std=0.0000
  nDCG@10          : mean=1.0000  std=0.0000
  MAP@5            : mean=0.4855  std=0.2301
  MAP@10           : mean=0.7681  std=0.1534

  effective_rank     : 1.16  (ratio=0.0726,  dim=16)

── per_class ───────────────────────────────────────────────────────────

  [corn]  n=16
    pairwise cosine  : mean=0.9681  std=0.0131
    centroid cosine  : mean=0.9849  norm=0.9849
    effective_rank   : 6.51  (ratio=0.4070)
    KNN purity@5     : mean=1.0000  std=0.0000  (p05=1.0000  p95=1.0000

## Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes, and drag to rotate 3D views.

### KNN Confusion Matrix

Rows = true class, columns = neighbor class, values = fraction of k-NN neighbors belonging to each class.  
The diagonal equals mean KNN purity — higher is better.

In [16]:
plot_knn_confusion(result, output_path=None)

### Pairwise Cosine Similarity

Full N×N cosine similarity matrix sorted by class.  Within-class blocks sit on the diagonal — tighter, brighter blocks indicate a more discriminative embedding space.

In [17]:
plot_cosine_similarity(embeddings, result, output_path=None)

### t-SNE — 2D

t-SNE preserves local neighborhood structure.  Well-separated clusters indicate the model has learned class-discriminative features.

In [18]:
plot_tsne(embeddings, result, output_path=None, dimensions=2)

### t-SNE — 3D

3D variant — drag to rotate, scroll to zoom.

In [19]:
plot_tsne(embeddings, result, output_path=None, dimensions=3)

### LLE — 3D

Locally Linear Embedding preserves local geometry rather than global distances, complementing the t-SNE view.  Drag to rotate.

In [20]:
plot_lle(embeddings, result, output_path=None)